# Pandas Part 4: Clustering and Cluster Dynamics

This notebook rebuilds the enriched feature set required for clustering, then covers KMeans, UMAP/PCA views, and the follow-up cluster-specific consequence analysis.

Links:
- [Pandas index](../data_analytics_project_pandas.ipynb)
- [Shared helpers](../functions.py)

In [ ]:
%load_ext autoreload
%autoreload 2
# TODO: Update it to the same level as the polars notebook

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start] + list(start.parents):
        if (candidate / 'data' / 'co2_data.csv').exists() and (candidate / 'notebooks' / 'functions.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
NOTEBOOKS_DIR = REPO_ROOT / 'notebooks'
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
DATA_DIR = REPO_ROOT / 'data'

In [ ]:
import pandas as pd
import numpy as np 

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay
import umap.umap_ as umap

import timeit
import requests
import json
from flatten_json import flatten

from functions import (
    apply_correlation_to_df,
    normalize_column,
    z_score_column,
    safe_divide,
    classify_income_group,
    get_top_bottom_n,
    compute_energy_mix_shares,
    plot_dual_axis_timeseries,
    run_kmeans_elbow,
)

# professional theme: clean white grid with muted palette
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "figure.titlesize": 16,
})
%matplotlib inline

## CO2 Emissions Data

We load the Our World in Data CO2 dataset, which contains 79 columns spanning energy mix, land use, and emissions breakdowns. For this analysis we reduce it to five key variables: `country`, `year`, `iso_code`, `population`, and `co2` (total production-based CO2 emissions in millions of tonnes).

Two important filtering steps:
- **Drop rows without `iso_code`**: The dataset includes aggregate entities like "Africa", "OECD", and "World" that lack ISO country codes. Removing these ensures we work exclusively with individual nation-states.
- **Filter to post-1960**: GDP data from the World Bank only begins in 1960, so we align the time range to enable a clean merge later.

In [ ]:
co2_df = pd.read_csv(DATA_DIR / 'co2_data.csv')

# choose what columns to keep (can be changed, but this is the most important)
selected_columns = ['country', 'year', 'iso_code', 'population', 'co2']

# drop the columns that are not in the selected_columns list
co2_df = co2_df[selected_columns].copy()

# remove entries with no iso code(Continents and other groups)
co2_df = co2_df[co2_df['iso_code'].notna() & (co2_df['iso_code'].str.strip() != '')]
co2_df = co2_df[co2_df['year'] > 1960]

co2_df


## GDP Data

The World Bank GDP dataset arrives in **wide format** - one column per year (1960, 1961, ..., 2024). This is convenient for spreadsheet viewing but incompatible with tidy-data principles needed for plotting and merging. We use `pd.melt()` to reshape it into long format with one row per country-year observation.

Key steps:
- **Melt** year columns into `year` (int) and `gdp` (numeric) columns
- **Rename** `Country Code` to `iso_code` to create a shared merge key with the CO2 dataset
- **Drop missing GDP values** - not all countries have GDP records for every year, particularly in earlier decades or for newly independent states

In [ ]:
# load GDP data
gdp_df = pd.read_csv(DATA_DIR / 'gdp_data.csv')

# reshape from wide to long format
# melt the year columns into rows
year_columns = [col for col in gdp_df.columns if col.isdigit()]
gdp_df = gdp_df.melt(
    id_vars=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
    value_vars=year_columns,
    var_name='year',
    value_name='gdp'
)

# convert year to integer and gdp to numeric
gdp_df['year'] = gdp_df['year'].astype(int)
gdp_df['gdp'] = pd.to_numeric(gdp_df['gdp'], errors='coerce')

# rename Country Code to iso_code for merging
gdp_df = gdp_df.rename(columns={'Country Code': 'iso_code'})

# keep only the columns we need
gdp_df = gdp_df[['iso_code', 'year', 'gdp']]

# remove rows with missing GDP values
gdp_df = gdp_df.dropna(subset=['gdp'])

gdp_df

## Merging the Datasets

We perform a **left join** of GDP onto the CO2 dataframe using `iso_code` and `year` as composite keys. A left join preserves every CO2 record and attaches GDP where available - countries or years without World Bank GDP data simply receive NaN. This is preferable to an inner join because it avoids silently discarding emission records that are still valuable for other analyses.

### Missing Data Heatmap

Before proceeding, we visualize data completeness. The heatmap below shows each variable as a column and each record as a row, with bright cells indicating missing values. This diagnostic is important because it reveals whether missingness is **random** or **systematic** - for instance, GDP data may be consistently absent for certain countries or time periods, which would bias any analysis that silently drops incomplete rows.

In [ ]:
# merge the datasets on iso_code and year
base_df = co2_df.merge(
    gdp_df,
    on=['iso_code', 'year'],
    how='left',
    suffixes=('_co2', '_gdp')
)

# missing data heatmap
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(base_df.isnull(), yticklabels=False, cbar=False, cmap='YlOrRd', ax=ax)
ax.set_title("Missing Data Overview After Merge")
plt.tight_layout()
plt.show()


### Data Quality Summary

The GDP column shows the most missingness - this is expected since many countries (especially newly independent or conflict-affected states) lack World Bank GDP records in earlier decades. Importantly, the missingness is **systematic** rather than random: it concentrates in specific countries and time periods. This means any analysis involving GDP will implicitly exclude these observations, so conclusions are most robust for the subset of countries with consistent economic data.

## Per-Capita Metrics

Absolute CO2 and GDP figures are dominated by population size - China and India will always top the charts simply because they have the most people, not necessarily because their economies or industries are more carbon-intensive on a per-person basis. Dividing by population yields **per-capita** values that enable fairer cross-country comparisons: how much does the average citizen emit, and how wealthy is the average citizen?

- **CO2 per capita** is expressed in **tonnes per person** (the raw CO2 column is in millions of tonnes, so we multiply by 10⁶ before dividing by population).
- **GDP per capita** is in **current USD per person**.

We use `np.divide` with a `where` guard to handle zero or missing population entries without raising division errors.

In [ ]:
# CO2 is in millions of tonnes; multiply by 1e6 to get tonnes, then divide by population
base_df['co2_per_capita'] = safe_divide(base_df['co2'].values * 1e6, base_df['population'].values)
base_df['gdp_per_capita'] = safe_divide(base_df['gdp'].values, base_df['population'].values)

# Log-transform per-capita metrics for better visualization
base_df['log_gdp_pc'] = np.log1p(base_df['gdp_per_capita'].values)
base_df['log_co2_pc'] = np.log1p(base_df['co2_per_capita'].values)

# compare raw and log-transformed data
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(base_df['gdp_per_capita'].dropna(), bins=50, edgecolor='white')
axes[0].set_title('GDP per Capita – Raw Data')
axes[1].hist(base_df['log_gdp_pc'].dropna(), bins=50, edgecolor='white', color='seagreen')
axes[1].set_title('GDP per Capita – Log-Transformed (approx. normal)')
plt.tight_layout()
plt.show()


## Income Group Classification

To move beyond individual country case studies, we classify each observation by **income group** using the World Bank's Gross National Income (GNI) thresholds:

| Group | GNI per capita (current USD) |
|---|---|
| Low | $\leq$ 1,145 |
| Lower-Middle | 1,146 – 4,515 |
| Upper-Middle | 4,516 – 14,005 |
| High | $>$ 14,005 |

Source: [World Bank Country Classification (FY2025)](https://datahelpdesk.worldbank.org/knowledgebase/articles/906519-world-bank-country-and-lending-groups)

This allows us to ask a structural question: do emission trajectories differ systematically between rich and poor countries? We scrape current GNI per capita figures from Wikipedia and merge them onto our dataset, then apply `pd.cut` to assign income brackets based on GDP per capita as a proxy.

In [ ]:
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_GNI_(nominal)_per_capita'

tables = pd.read_html(url, storage_options={'User-Agent': 'Mozilla/5.0'})

# table 1 contains the GNI per capita data
gni_df = tables[1]

dict = {
    'GNI per capita (US$)[1][3]': 'gni_per_capita'
}

gni_df = gni_df.rename(columns=dict)

gni_df



In [ ]:
# normalize 
gni_df['country_clean'] = normalize_column('Country', gni_df)
base_df['country_clean'] = normalize_column('country', base_df)

# left merge
main_df = pd.merge(base_df, gni_df[['country_clean', 'gni_per_capita']], 
                     on='country_clean', how='left')

# classify income groups using World Bank thresholds
main_df['income_group'] = classify_income_group(main_df['gni_per_capita'])

main_df

In [ ]:
# Carbon intensity setup extracted from the original timing cell.
# carbon intensity: CO2 (millions of tonnes) per $1M GDP
gdp_in_millions = main_df['gdp'].values / 1_000_000
main_df['co2_per_gdp'] = safe_divide(main_df['co2'].values, gdp_in_millions)

In [ ]:
# fetch electricity production data from Our World in Data
df_electricity = pd.read_csv(
    "https://ourworldindata.org/grapher/electricity-prod-source-stacked.csv?v=1&csvType=full&useColumnShortNames=true",
    storage_options={'User-Agent': 'Our World In Data data fetch/1.0'}
)

# rename to concise column names
rename_map = {
    'code': 'iso_code',
    "other_renewables_excluding_bioenergy_generation__twh_chart_electricity_prod_source_stacked": "other_renewables",
    'bioenergy_generation__twh_chart_electricity_prod_source_stacked': 'bioenergy',
    'solar_generation__twh_chart_electricity_prod_source_stacked': 'solar',
    'wind_generation__twh_chart_electricity_prod_source_stacked': 'wind',
    'hydro_generation__twh_chart_electricity_prod_source_stacked': 'hydro',
    'nuclear_generation__twh_chart_electricity_prod_source_stacked': 'nuclear',
    'oil_generation__twh_chart_electricity_prod_source_stacked': 'oil',
    'gas_generation__twh_chart_electricity_prod_source_stacked': 'gas',
    'coal_generation__twh_chart_electricity_prod_source_stacked': 'coal',
}
df_electricity = df_electricity.rename(columns=rename_map)

# keep only individual countries (drop aggregates without ISO codes)
df_electricity = df_electricity[df_electricity['iso_code'].notna() & (df_electricity['iso_code'].str.strip() != '')]

print(f"Electricity data: {df_electricity.shape[0]:,} rows, "
      f"{df_electricity['iso_code'].nunique()} countries, "
      f"{df_electricity['year'].min()}-{df_electricity['year'].max()}")

df_electricity

In [ ]:
green_cols = ['other_renewables', 'bioenergy', 'solar', 'wind', 'hydro', 'nuclear']
non_green_cols = ['coal', 'oil', 'gas']

df_electricity = compute_energy_mix_shares(df_electricity, green_cols, non_green_cols)

In [ ]:
# columns coming from df_electricity (excluding merge keys and entity)
elec_cols = [c for c in df_electricity.columns if c not in ['entity', 'iso_code', 'year']]

# drop these from base_df if they already exist (handles notebook re-runs)
base_df = base_df.drop(columns=[c for c in elec_cols if c in base_df.columns])

# merge electricity data onto base_df
base_df = base_df.merge(
    df_electricity.drop(columns=['entity']),
    on=['iso_code', 'year'],
    how='inner'
)

base_df

# Unsupervised Clustering: Country Archetypes

The analyses above used predefined income brackets to group countries. But do countries **naturally** cluster into distinct profiles when we let the data decide? We apply **KMeans clustering** on three features from the most recent year (2023):

1. **GDP per capita** - economic wealth
2. **CO2 per $1M GDP** - carbon intensity of the economy
3. **Non-green electricity share** - fossil fuel dependency in the power sector

All features are standardised (z-scored) before clustering to prevent GDP per capita (measured in thousands of USD) from dominating the distance metric over shares (measured between 0 and 1).

In [ ]:
year = 2023

ml_features = ['log_gdp_pc','log_co2_pc', 'co2_per_gdp', 'non_green_share']
cluster_df = base_df[base_df['year'] == year].dropna(subset=ml_features).copy()

# run kmeans elbow ( in numpy)
X = cluster_df[ml_features].to_numpy()

# run kmeans elbow
fig, ax = plt.subplots(figsize=(8, 5))
_, scaled_data, _, inertias, scaler = run_kmeans_elbow(
    X,
    k_range=range(1, 8),
    scale=True,
    random_state=42,
    n_init=10,
    n_clusters=None,
    plot=True,
    ax=ax,)
plt.tight_layout()

### Choosing the Number of Clusters

The elbow plot shows inertia (within-cluster sum of squares) for k = 1 through 7. The "elbow" at **k = 3** indicates diminishing returns beyond four clusters, making it the optimal choice for segmenting countries by economic and environmental characteristics.

In [ ]:
# Cluster countries
cluster_labels, scaled_data, kmeans_model, inertias, scaler = run_kmeans_elbow(
    X,
    k_range=range(1, 8),
    n_clusters=3,
    plot=False,
    ax=None,
    scale=True,
    random_state=42,
    n_init=10,)

# Add cluster labels to the original data
cluster_df["cluster"] = cluster_labels

cluster_df

In [ ]:
# create a pairplot of scatterplots
g = sns.pairplot(cluster_df, vars=ml_features, hue='cluster', palette='bright',
                 diag_kind='kde', plot_kws={'alpha': 0.7, 'edgecolor': 'none'})
                 
g.fig.suptitle(f'Country Clusters: Wealth, Carbon Intensity & Energy Mix ({year})', y=1.02)
plt.show()

### Interpreting the Pairplot

The pairplot visualizes pairwise relationships between the three clustering features - GDP per capita, CO2 per $M GDP, and non-green electricity share - with each point colored by its KMeans cluster assignment.

- **GDP per capita vs. CO2 per GDP** shows a clear inverse relationship: wealthier countries tend to emit far less CO2 per unit of economic output, reinforcing the efficiency narrative from earlier sections.
- **Non-green share vs. CO2 per GDP** correlates positively - countries that rely heavily on fossil fuels for electricity are also the least carbon-efficient economies overall.
- **GDP per capita vs. non-green share** is more dispersed, suggesting that wealth alone does not guarantee a clean electricity mix (e.g. oil-rich Gulf states are wealthy but fossil-dependent).

The KDE diagonals reveal that most clusters are well-separated along at least one axis, confirming that the three-cluster solution captures meaningful structural differences rather than arbitrary splits.

In [ ]:
base_df

# Further cluster analysis
Let's have a look at the clusters in detaiö, finding out what exactly is behind the clusters and how effective kmeans was

In [ ]:
ml_features = ["log_gdp_pc", "log_co2_pc", "co2_per_gdp", "non_green_share", "cluster"]

# clean data
cluster_df.dropna()

# scale data
X = cluster_df[ml_features[0:4]].to_numpy()

X_scaled = StandardScaler().fit_transform(X)

# reduce dimensions
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=42,)

# transform data
embedding = reducer.fit_transform(X_scaled)

cluster_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# plot UMAP embeddings for each feature
for ax, feature in zip(axes.flatten(), ml_features):
    sc = ax.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=cluster_df["cluster"],
        cmap="viridis",
        s=30)
    ax.set_title(feature)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    
plt.tight_layout()
plt.show()

In [ ]:
# extract main dimension (PCA)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# scatter them 
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=cluster_df["cluster"],
    cmap="viridis",
    s=30)
    
plt.colorbar(label="log_gdp_pc")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Projection (colored by Cluster)")
plt.show()

### UMAP analysis 
The UMAP visualization reveals a clear tension between economic growth and environmental sustainability. There is a visible positive correlation between wealth and emissions, as the clusters with the highest GDP per capita also show the highest carbon footprints. In contrast, the green share and non-green share plots act as near-perfect inverses; the points with the most significant sustainable activity are currently clustered in the lower-wealth regions. Ultimately, the data highlights a significant sustainability gap: the top-right clusters represent high-income but high-polluting entities, while the bottom-left reflects a greener but less wealthy profile, suggesting that a high-GDP, high-green-share model has not yet become a dominant reality.

In [ ]:
# extract main dimension
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=cluster_df["log_gdp_pc"],
    cmap="viridis",
    s=30
)
plt.colorbar(label="log_gdp_pc")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Projection (colored by GDP)")
plt.show()

### Comparison to UMAP
Compared to the UMAP projection, which reveals a nonlinear manifold structure, the PCA projection appears more compressed and less structured. This highlights the limitation of PCA in capturing nonlinear relationships, as it can only represent linear combinations of features.

In [ ]:
pca.components_

### More than just GDP?

In order to fact-check the conclusion of my PCA plot I decided to look at the values indivudually. We can actually see that the model trained more than purely GDP. It detected two dimensions:<br>
-**Wealth**: Cluster one is deemed welthy and efficient <br>
-**Energy and emmision efficiency**: Developmental countries like cluster 0 are low-income and inefficient, whereas cluster two depict very carbon-intensive economies



In [ ]:
fig, (ax,ax2) = plt.subplots(1, 2,figsize=(16,8))

sns.boxplot(data=cluster_df, x="cluster", y="non_green_share", ax=ax)
sns.boxplot(data=cluster_df, x="cluster", y="co2_per_gdp", ax=ax2, color='red')
plt.show()

In [ ]:
# look at countries in cluster 2
cluster_df[cluster_df["cluster"] == 2].sort_values("co2_per_gdp", ascending=False)

You could have a look at the least efficient economies from earlier: They're pretty much the same. The model actually found a useful economical structure without being given earlier information.

Let's have a look at the efficiency of cluster 2 in comparison to cluster 0 and 1

In [ ]:
fig, (ax1,ax2) = plt.subplots(1,2, figsize=(16,8))

# only look at cluster 2
cluster2 = cluster_df[cluster_df["cluster"] == 2]

# scatter
sns.scatterplot(
    data=cluster2,
    x="non_green_share",
    y="co2_per_gdp",
    ax=ax1)

sns.scatterplot(
    data=cluster_df,
    x="log_gdp_pc",
    y="co2_per_gdp",
    hue="cluster",
    ax=ax2
)

plt.tight_layout()
plt.show()

This reaffirms the suspicions, that cluster two is a real economic phenomenon with low-efficiency economies, mostly being run on fossil fuels. Some countries in cluster two like Laos or Vietnam might not have a high non_green share, however this might be explainable by their post-communist/communist, low-efficient economies.

Therefore we can say, that the Carbon intensity of economy is not mainly determined by economies. Structural differences like the energy-mix, industry-profile also play a significant role.


### What are the consequences?

A major driving factor for the climate change are said low-income, high-intensity economies we've looked at. Economies that see a high correlation between carbon emmisions and economic growth with little to no decoupling. 

I'm sure we are all aware of the consequences but let's also have a look at the global climate.

In [ ]:
# get the data from the internet
url = "https://nyc3.digitaloceanspaces.com/owid-public/data/co2/owid-co2-data.json"
response = json.loads(requests.get(url).text)

response

In [ ]:

# empty list for flattened dataset
rows = []

for country, content in response.items():
    iso_code = content.get('iso_code')
    for entry in content.get('data', []):
        # vreating a copy and adding iso_codes
        flattened_entry = entry.copy()
        flattened_entry['country'] = country
        flattened_entry['iso_code'] = iso_code
        rows.append(flattened_entry)

# create dataframe
json_df = pd.DataFrame(rows)

# sort the columns
cols = ['country', 'iso_code', 'year'] + [c for c in json_df.columns if c not in ['country', 'iso_code', 'year']]
json_df = json_df[cols]

# convert years to int and only take values from 1970 onwards
json_df["year"] = json_df["year"].astype(int)
json_df = json_df[json_df["year"] >= 1970]

json_df

In [ ]:
# filter the json df for all the countries in cluster 2
cluster_2_json = json_df[json_df["iso_code"].isin(cluster2["iso_code"])]

cluster_2_json

In [ ]:
# Create a dataframe for cluster 2 only by aggregating all values per year
cluster_2_per_year = cluster_2_json.groupby("year").sum(numeric_only=True).reset_index()
cluster_2_per_year["Group"] = "Cluster 2"

cluster_2_per_year

In [ ]:
# Plotting the primary drivers
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.set_xlabel('Year')
ax1.set_ylabel('Population', color='tab:blue')
ax1.plot(cluster_2_per_year['year'], cluster_2_per_year['population'], color='tab:blue', label='Population')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.set_ylabel('Cement CO2 per Capita', color='tab:red')
ax2.plot(cluster_2_per_year['year'], cluster_2_per_year['cement_co2_per_capita'], color='tab:red', label='Cement CO2 per Cap')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title('Cluster 2: Population Growth vs. Emission Intensity')
plt.show()

We can observe a stagnating Cement CO2 per Capita value between 1970 and 1995. Afterwards there's a rapid incline, mainly due to increasing industrialisation of emerging economies like China with a steady and almost linear increase of population.

In [ ]:
# Plotting the primary drivers
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.set_xlabel('Year')
ax1.set_ylabel('Population', color='tab:blue')
ax1.plot(cluster_2_per_year['year'], cluster_2_per_year['share_global_other_co2'], color='tab:blue', label='Population')
ax1.tick_params(axis='y', labelcolor='tab:blue')

plt.title('Cluster 2: Other Co2 Share')
plt.show()

It is also worth noting, that cluster 2 accounts for roughly 50-65% of global co2 share since 1990. This shows the intensity of those economies, which is steadily climbing. This could also indicate that other economies, like the rich ones, achieve greater decoupling, slowly increasing Cluster 2s share of Co2 emmisions.

In [ ]:
world_df = json_df[json_df["country"] == "World"].groupby("year").sum(numeric_only=True)

world_df